### Includes and def functions

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch 
import torch.nn as nn
from blackcatt.task import *
from blackcatt.wm_task import *
from blackcatt.models import *
from tqdm import tqdm
from random import sample
from torchvision.transforms import Compose, Normalize, ToTensor, RandomCrop, RandomHorizontalFlip, Lambda
from PIL import Image
from math import comb
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Initializing parameters and datasets

In [ ]:
# CIFAR-10 (10 classes)
# m = 250
# n_users = 20
# clients_tardos_q = np.loadtxt("wm_constants/" + 'clients_tardos_q_10_k0.5.csv').astype(int)[:n_users,:m]
# p_secret = np.loadtxt("wm_constants/" + 'p_secret_10_k0.5.csv').astype(float)[:m,:]
# tau =  0.01

# CIFAR-100 (100 classes)
m = 250
n_users = 20
clients_tardos_q = np.loadtxt("wm_constants/" + 'clients_tardos_q_100_k0.5.csv').astype(int)[:n_users,:m]
p_secret = np.loadtxt("wm_constants/" + 'p_secret_100_k0.5.csv').astype(float)[:m,:]
tau =  0.001

In [ ]:
trainloader, valloader = load_data(0, 20, "CIFAR100")

# Train loaders for all data owners
all_training_loaders = []
for i in range(20):
    trainloader, testloader = load_data(i, 20, "CIFAR100")
    all_training_loaders.append(trainloader)

### Selecting experiments

In [ ]:
# Load the experiments
experiments = pd.read_csv('all_exp.csv')
folders_index = experiments['folder'].tolist()
labels_index = experiments['label'].tolist()
colors_index = experiments['color'].tolist()
triggers_path_index = experiments['triggers_path'].tolist()

In [ ]:
# VGG
# index = [354,429,420]
# ResNet
index = [371,430,434,425]
labels = ["No WM","Vanilla","BlackCATT", "BlackCATT+FR"]
# index = [430,434,425]
# labels = ["Vanilla","BlackCATT", "BlackCATT+FR"]
# Number triggers
# index = [432,425,442]
# labels = [r"$M=100$", r"$M=250$", r"$M=500$"]
# Number DOs
# index = [425,428,431]
# labels = [r"$N=20$", r"$N=40$", r"$N=60$"]
# Optimization rounds
# index = [425,426,424]
# labels = ["1 round","2 rounds","5 rounds"]
# Effect of aux dataset
# index = [441,440,437,439,438]
# labels = ["No WM", "BlackCATT", "WikiArt", "CIFAR-100", "TinyImageNet"]
# index = [441,440]
# labels = ["No WM", "BlackCATT"]
# Ablation
# index = [425,434,436,435]
# labels = ["BlackCATT+FR", "BlackCATT", r"BlackCATT+FR w/o $\nabla_{\mathbf{x}^{(r)}}$", r"BlackCATT+FR w/o $L_\text{CA}$"]
# index = [425,436,434,435]
# labels = ["BlackCATT+FR", r"BlackCATT+FR w/o $\nabla_{\mathbf{x}^{(r)}}$", "BlackCATT", r"BlackCATT+FR w/o $L_\text{CA}$"]
# Benign triggers
# index = [371,425,433]
# labels = ["No WM", "BlackCATT+FR", "BlackCATT(B9)+FR"]

folders = [folders_index[i] for i in index]
colors = [colors_index[i] for i in index]
triggers_path = [triggers_path_index[i] for i in index]

### Computing metrics

#### Accusation to csv for all selected experiments
Note the selected network and initialized parameters should match the experiment (label vector and p bias path, number of triggers, tau)

In [ ]:
# net = VGG16(num_classes=10)
net = ResNet18(num_classes=100)
net.to(device)
net.eval()

fn = []
n_collusions = 100
max_colluders = 6
collusion_strategy = "average" # "average" or "randomselect"

for folder in folders:
    print("Loading folder: ",folder)
    print(triggers_path[folders.index(folder)])

    try:
        df_all = pd.read_csv("df_all_"+collusion_strategy+"_"+str(index[folders.index(folder)])+".csv")  
    except:
        print("File not found, creating new one.")
        df_all = pd.DataFrame({"strategy":[],"c":[],"m_needed":[],"fn":[]})

    triggers = np.load(triggers_path[folders.index(folder)])
    triggers_transforms = Compose(
        [ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
    )
    triggers = torch.stack([triggers_transforms(Image.fromarray(trigger.astype(np.uint8))) for trigger in triggers]).float().to(device)
  
    fn_i_average = []
    for c in tqdm(range(1,max_colluders+1)):
        fn = []
        m_needed = []
        for i in range(n_collusions):   
            net.eval()          
            # Choose random colluders
            colluders = sample(list(range(20)),c)
            # Load the merged model according to the collusion strategy
            net = load_collusion(colluders,net,folder,mode=collusion_strategy,device=device)

            output = net(triggers)

            y = torch.argmax(output, dim=1).cpu().numpy()

            tp, t_s = tardos_accusation(y,vectors=clients_tardos_q, p_secret=p_secret, tau=tau)
            if tp==-1:
                # No accusation made
                fn.append(1)
                m_needed.append(triggers.shape[0])
            elif tp in colluders:
                fn.append(0)
                m_needed.append(t_s)
            else:
                print("False positive detected in colluders:", colluders, " accused:", tp)

        df = pd.DataFrame({"strategy":[labels[folders.index(folder)] for _ in range(len(m_needed))],
            "c":[c for _ in range(len(m_needed))],
            "m_needed":m_needed,
            "fn":fn})
        df_all = pd.concat([df_all,df])

    df_all.to_csv("df_all_"+collusion_strategy+"_"+str(index[folders.index(folder)])+".csv",index=False)

#### Fine-pruning
Note the selected network, datasets, and initialized parameters should match the experiment (label vector and p bias path, number of triggers, tau)

In [ ]:
fn = []
n_collusions = 17
max_colluders = 6
pruning_rates = [0.0, 0.25, 0.5, 0.75, 0.85, 0.95]

# Only for one experiment at a time
index = [425]
folders = [folders_index[i] for i in index]
labels = [labels_index[i] for i in index]
colors = [colors_index[i] for i in index]
triggers_path = [triggers_path_index[i] for i in index]

try:
    df_all = pd.read_csv("df_all_fine_prun_"+str(index[0])+"_new.csv")  
except:
    print("File not found, creating new one.")
    df_all = pd.DataFrame({"pruning_rate":[],"c":[],"accuracy":[],"fn":[],"m_needed":[]})

print("Experiment: ",labels[0])
print("Loading folder: ",folders[0])
triggers = np.load(triggers_path[0])
triggers_transforms = Compose(
    [ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)
triggers = torch.stack([triggers_transforms(Image.fromarray(trigger.astype(np.uint8))) for trigger in triggers]).float().to(device)

for i in range(n_collusions):    
    for pruning_rate in pruning_rates:
        print("Pruning rate: ",pruning_rate)
        fn_i_average = []
        for c in tqdm(range(1,max_colluders+1)):
            fn_i = 1
            m_needed = []
            # net = VGG16(num_classes=10)
            net = ResNet18(num_classes=100)
            net.to(device)
            net.eval()

            # Choose random colluders
            colluders = sample(list(range(20)),c)
            # Gather fine-tuning dataloaders
            fine_dataloaders = [all_training_loaders[k] for k in colluders]
            # Load the merged model and fine-tune with pruning
            net = load_collusion(colluders,net,folders[0],device=device,fine=5,fine_dataloader=fine_dataloaders,ft_lr=0.01,pruning=pruning_rate)
            # Test main task accuracy of attacked model
            net.eval()       
            loss, acc = test(net, testloader, device)
            output = net(triggers)

            y = torch.argmax(output, dim=1).cpu().numpy()

            tp, t_s = tardos_accusation(y,vectors=clients_tardos_q, p_secret=p_secret, tau=tau)
            if tp==-1:
                # No accusation made
                fn_i = 1
            elif tp in colluders:
                fn_i = 0
            else:
                print("False positive detected in colluders:", colluders, " accused:", tp)

            df = pd.DataFrame({"pruning_rate":[pruning_rate],
                            "c":[c],
                            "accuracy":[acc],
                            "fn":[fn_i],
                            "m_needed":[t_s]})
            df_all = pd.concat([df_all,df])

            df_all.to_csv("df_all_fine_prun_"+str(index[0])+"_new.csv",index=False)  


#### Matched / Mismatched snapshot
Note the selected network and initialized parameters should match the experiment (label vector and p bias path, number of triggers, tau)

In [ ]:
# net = VGG16(num_classes=10)
net = ResNet18(num_classes=100)
net.to(device)
net.eval()

n_collusions = 100
index = 426
folder = folders_index[index]

try:
    df_all = pd.read_csv("df_all_matched_"+str(index)+".csv")  
except:
    print("File not found, creating new one.")
    df_all = pd.DataFrame({"trigger":[],"snapshot":[],"c":[],"m_needed":[],"fn":[]})

for trigger_i in tqdm([250,500,1000,1500]):

    try:
        triggers = np.load(folder + "trigger_round_" + str(trigger_i) + ".npy")
        triggers_transforms = Compose(
            [ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
        )
        triggers = torch.stack([triggers_transforms(Image.fromarray(trigger.astype(np.uint8))) for trigger in triggers]).float().to(device)
    except:
        print(trigger_i,"not found")
        break
    
    for snap_i in tqdm([250,500,1000,1500]):

        for c in [2,5]:
                for i in range(n_collusions):
                    colluders = sample(list(range(20)),c)

                    net = load_collusion(colluders,net,folder + str(snap_i) + "_")
                
                    output = net(triggers)
                    y = torch.argmax(output, dim=1).cpu().numpy()

                    tp, t_s = tardos_accusation(y,vectors=clients_tardos_q, p_secret=p_secret, tau=tau)
                    if tp==-1:
                        # No accusation made
                        fn_i = 1
                        m_needed = triggers.shape[0]
                    elif tp in colluders:
                        fn_i = 0
                        m_needed = t_s
                    else:
                        print("False positive detected in colluders:", colluders, " accused:", tp)

                    df = pd.DataFrame({"trigger":[trigger_i],
                        "snapshot":[snap_i],
                        "c":[c],
                        "m_needed":[m_needed],
                        "fn":[fn_i]})
                    df_all = pd.concat([df_all,df])

df_all.to_csv("df_all_matched_"+str(index)+".csv",index=False)  

#### Experimental FPR for wrong data-owner

In [ ]:
# All experiments together
# BlackCATT+FR
# index = [425,432,418,428,431,426,424,433,437,439,438]
# BlackCATT
# index = [420,434,440]
index = [428,431]

labels = [labels_index[i] for i in index]
folders = [folders_index[i] for i in index]
colors = [colors_index[i] for i in index]
triggers_path = [triggers_path_index[i] for i in index]
print(len(index))
pfp = 0.01
n_collusions = 100

try:
    df_all = pd.read_csv("df_all_pfp"+str(pfp)+".csv")  
except:
    print("File not found, creating new one.")
    df_all = pd.DataFrame({"index":[],"net":[],"dataset":[],"m":[],"n":[],"fr":[],"c":[],"m_needed":[],"fn":[],"fp":[]})

for i, folder in tqdm(enumerate(folders)):
    # Initializing everything for the specific experiment
    index_i = index[i]
    try:
        m_i = int((folder.split("m_")[0].split("_")[-1]))
    except:
        m_i = int((folder.split("mB9_")[0].split("_")[-1]))
    n_i = int((folder.split("users_")[0].split("_")[-1]))
    if "CIFAR100" in folder:
        dataset_i = "CIFAR100"
        clients_tardos_q = np.loadtxt("wm_constants/" + 'clients_tardos_q_100_k0.5.csv').astype(int)[:,:m_i]
        p_secret = np.loadtxt("wm_constants/" + 'p_secret_100_k0.5.csv').astype(float)[:m_i,:]
        tau =  0.001
    else:
        dataset_i = "CIFAR10"
        clients_tardos_q = np.loadtxt("wm_constants/" + 'clients_tardos_q_10_k0.5.csv').astype(int)[:,:m_i]
        p_secret = np.loadtxt("wm_constants/" + 'p_secret_10_k0.5.csv').astype(float)[:m_i,:]
        tau =  0.01
    if "CLAvgKL0.1" in folder:
        fr_i = True
    else:
        fr_i = False
    if "ResNet" in folder:
        net_i = "ResNet18"
        if dataset_i == "CIFAR100":
            net = ResNet18(num_classes=100)
        else:
            net = ResNet18(num_classes=10)
    else:
        net_i = "VGG16"
        net = VGG16()
    net.to(device)
    net.eval()     

    # Prepare specific triggers
    triggers = np.load(triggers_path[i])
    triggers_transforms = Compose(
        [ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
    )
    triggers = torch.stack([triggers_transforms(Image.fromarray(trigger.astype(np.uint8))) for trigger in triggers]).float().to(device)

    # Check collusions
    for c in [1,2,5]:
        fn = []
        fp = []
        m_needed = []
        if n_collusions > comb(n_i,c):
            n_collusions_i = comb(n_i,c)
        else:
            n_collusions_i = n_collusions
        for i_col in range(n_collusions_i):     
            try:
                if c == 1:
                    colluders = [i_col]
                else:
                    colluders = sample(list(range(n_i)),c)

                net = load_collusion(colluders,net,folder,mode="average",device=device)

                output = net(triggers)

                y = torch.argmax(output, dim=1).cpu().numpy()
                
                tp, t_s = tardos_accusation(y,vectors=clients_tardos_q, p_secret=p_secret, tau=tau,pfp=pfp)
                if tp==-1:
                    fn.append(1)
                    fp.append(0)
                    m_needed.append(triggers.shape[0])
                elif tp not in colluders:
                    fn.append(0)
                    fp.append(1)
                    m_needed.append(t_s)
                else:
                    fn.append(0)
                    fp.append(0)
                    m_needed.append(t_s)
            except:
                pass

        df = pd.DataFrame({"index":[index_i for _ in range(len(m_needed))],
            "net":[net_i for _ in range(len(m_needed))],
            "dataset":[dataset_i for _ in range(len(m_needed))],
            "m":[m_i for _ in range(len(m_needed))],
            "n":[n_i for _ in range(len(m_needed))],
            "fr":[fr_i for _ in range(len(m_needed))],
            "c":[c for _ in range(len(m_needed))],
            "m_needed":m_needed,
            "fn":fn,
            "fp":fp})
        df_all = pd.concat([df_all,df])

        df_all.to_csv("df_all_pfp"+str(pfp)+".csv",index=False)


In [ ]:
# Print experimental FPR according to c and n
df_all = pd.read_csv("df_all_pfp0.01.csv")
avg_fp = df_all.groupby(["c", "n"])["fp"].mean().reset_index(name="avg_fp")
pivot_avg_fp = avg_fp.pivot(index="n", columns="c", values="avg_fp")

print(pivot_avg_fp)

#### Effect of trigger optim iterations

In [ ]:
df_all = pd.DataFrame({"strategy":[],"round":[],"loss":[],"accuracy":[],"m_accuracy":[],"trigger":[],"mav":[],"fn":[]})

# Read the metrics file
for folder in folders:
    # print("Processing folder:", folder)
    for i_cid in range(10):
        # try:
            with open(folder + "metrics_"+str(i_cid)+".csv", "r") as file:
                data = file.readlines()
                max_acc = 0
                for round, line in enumerate(data):
                    loss_i = float(line.split(",")[0])
                    accuracy_i = float(line.split(",")[1])
                    accuracy_t_i =  float(line.split(",")[2])
                    if accuracy_i > max_acc:
                        max_acc = accuracy_i
                    mav_i =  float(line.split(",")[3])
                    try:
                        fn_i =  float(line.split(",")[4] =="True")
                    except:
                        # print("No fn info")
                        fn_i = 1
                    df = pd.DataFrame({"strategy":[labels[folders.index(folder)]],
                        "round":[(round+1)*25],
                        "loss":[loss_i],
                        "accuracy":[accuracy_i],
                        "m_accuracy":[max_acc],
                        "trigger":[accuracy_t_i],
                        "mav":[mav_i],
                        "fn":[fn_i]})
                    df_all = pd.concat([df_all,df])

        # except:
        #     pass
        
thr_mav = []
# Calculate threshold MAV for each strategy
for i, folder in enumerate(folders):
    strategy_data = df_all[df_all["strategy"] == labels[i]]
    
    # Group by round and calculate mean MAV for each round
    avg_fn_by_round = strategy_data.groupby("round")["fn"].mean()
    
    # Find first round where average MAV goes below 0.5 FNR
    threshold_round = None
    for round_num, avg_fn in avg_fn_by_round.items():
        if avg_fn < 0.5:
            threshold_round = round_num
            break
    
    thr_mav.append(threshold_round)

print("Threshold MAV rounds for each strategy:", thr_mav)
print("Mean acc @ threshold")
for i, folder in enumerate(folders):
    strategy_data = df_all[df_all["strategy"] == labels[i]]
    threshold_round = thr_mav[i]
    acc_at_threshold = strategy_data[strategy_data["round"] == threshold_round]["accuracy"].mean()
    print(f"{labels[i]}:{acc_at_threshold}")
print("Max final acc")
for i, folder in enumerate(folders):
    strategy_data = df_all[df_all["strategy"] == labels[i]]
    max_final_acc = strategy_data[strategy_data["round"] == strategy_data["round"].max()]["m_accuracy"].mean()
    print(f"{labels[i]}:{max_final_acc}")

#### Matched / Mismatched

In [ ]:
index = 426
folder = folders_index[index]

df_all = pd.read_csv("df_all_matched_"+str(index)+".csv")  

df_all = df_all[df_all["trigger"].isin([250,500,1000,1500])]
df_all = df_all[df_all["snapshot"].isin([250,500,1000,1500])]

df_all_2 = df_all[df_all["c"]==2]
avg_fn = df_all_2.groupby(["trigger", "snapshot"])["fn"].mean().reset_index(name="avg_fn")

# Optional: pivot for a matrix view (rows=trigger, cols=snapshot)
pivot_avg_fn = avg_fn.pivot(index="snapshot", columns="trigger", values="avg_fn")

# Show and save
# print(avg_fn.head())
print(pivot_avg_fn)            # prints matrix

df_all_6 = df_all[df_all["c"]==5]
avg_fn = df_all_6.groupby(["trigger", "snapshot"])["fn"].mean().reset_index(name="avg_fn")

# Optional: pivot for a matrix view (rows=trigger, cols=snapshot)
pivot_avg_fn = avg_fn.pivot(index="snapshot", columns="trigger", values="avg_fn")

# Show and save
# print(avg_fn.head())
print(pivot_avg_fn)            # prints matrix

In [ ]:
df_all_aux = df_all[df_all["strategy"]==labels[-2]]

print("Average Accuracy @ round 250:", df_all_aux[df_all_aux["round"]==250]["accuracy"].mean())
print("Average Accuracy @ round 500:", df_all_aux[df_all_aux["round"]==500]["accuracy"].mean())
print("Average Accuracy @ round 1000:", df_all_aux[df_all_aux["round"]==1000]["accuracy"].mean())
print("Average Accuracy @ round 1500:", df_all_aux[df_all_aux["round"]==1500]["accuracy"].mean())

#### Experimental FPR

In [ ]:
# Read all files starting with "df_all_pfp0.01_431_cNoWM" and ending with ".csv" and concatenate them into a single dataframe
import os
import pandas as pd
df_all = pd.DataFrame()

# Get a list of all files in the current directory
files = os.listdir("../../FL_models/new_version/")
for file in files:
    if file.startswith("df_all_pfp0.01_431_cNoWM") and file.endswith(".csv"):
        df_i = pd.read_csv(os.path.join("../../FL_models/new_version/", file))
        try:
            df_all = pd.concat([df_all, df_i])
        except:
            df_all = df_i

print(df_all["fp"].mean())
print(df_all[df_all["fp"] > 0]["fp"].count())
print(len(df_all.index))

#### Effect of collusion on unique triggers

In [ ]:
net = VGG16()
net.to(device)
net.eval()

n_collusions = 20
index = 381
folder = folders_index[index]

try:
    df_all = pd.read_csv("df_all_unique_"+str(index)+".csv")  
except:
    print("File not found, creating new one.")
    df_all = pd.DataFrame({"strategy":[],"c":[],"accuracy_t":[],"acc":[]})

print("Loading folder: ",folder)

X_trigger = np.loadtxt("wm_constants/" + 'X_trigger_unique_3x32x32.csv').astype(float)
X_trigger = np.rint(X_trigger*255)/255
X_trigger = X_trigger[:100*20]
X_trigger_tensor_full = np.reshape(X_trigger,[100*20,3,32,32])
X_trigger_tensor_full = torch.from_numpy(X_trigger_tensor_full).float()
# Normalize triggers
X_trigger_tensor_full = torch.stack([Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))(trigger) for trigger in X_trigger_tensor_full]).float()
X_trigger_tensor_full, labels_full = X_trigger_tensor_full.to(device), wm_config.clients_tardos_q_tensor.to(device)

strategies = ["No Attacks", "Fine-tuning", "Pruning and\nFine-tuning", "Averaging\nTwo Models"]
for strategy in strategies:
    if strategy == "No Attacks":
        fine = 0
        fine_dataloader = None
        pruning = 0
        c = 1
    elif strategy == "Fine-tuning":
        fine = 5
        fine_dataloader = trainloader
        pruning = 0
        c = 1
    elif strategy == "Pruning and\nFine-tuning":
        fine = 5
        fine_dataloader = trainloader
        pruning = 0.92
        c = 1
    elif strategy == "Averaging\nTwo Models":
        fine = 0
        fine_dataloader = None
        pruning = 0
        c = 2

    for i in range(n_collusions):   
        net.to(device)
        net.eval()          
        if c == 2:
            all = list(range(20))
            all.pop(i)
            col_i = sample(all, 1)
            colluders = [i, col_i[0]]
        else:
            colluders = [i]
        # Select triggers
        X_trigger_tensor = X_trigger_tensor_full[i *100:(i +1)*100]

        net = load_collusion(colluders,net,folder,device=device,fine=fine,fine_dataloader=fine_dataloader,ft_lr=0.01,pruning=pruning)
        loss, acc = test(net, testloader, device)
        
        output = net(X_trigger_tensor)
        _, predicted = output.max(1)
        correct = predicted.eq(labels_full[i]).sum().item()
        t_accuracy =correct/100

        df = pd.DataFrame({"strategy":[strategy],"c":[c],"accuracy_t":[t_accuracy],"acc":[acc]})
        df_all = pd.concat([df_all,df])

df_all.to_csv("df_all_unique_"+str(index)+".csv",index=False)  

#### Generating p bias and label vectors

In [ ]:
n_classes = 10
n_users = 100
n_segments = 1000
k = 0.5
tau = 0.01

p_secret = np.random.default_rng().dirichlet(np.ones(n_classes)*k,n_segments)
p_secret = np.array([ p_secret[i] * (1 - tau * (n_classes)) + tau for i in range(n_segments) ])

np.savetxt("./wm_constants/p_secret_10_k0.5.csv", p_secret)

clients_tardos_q = np.array([[np.random.choice(n_classes,p=p_secret[i]) for i in range(n_segments)] for _ in range(n_users)])

np.savetxt("./wm_constants/clients_tardos_q_10_k0.5.csv", clients_tardos_q)